## Read Silver

In [0]:
from pyspark.sql.functions import (
    col, 
    sum,
    round, 
    when
)

from pyspark.sql.window import Window

silver_table = "ev_registration.silver.silver_vehicle_registrations" 

gold_state_source_df = spark.read.table(silver_table)

print("Silver records:", gold_state_source_df.count())

display(gold_state_source_df.limit(10))

Silver records: 499995


registration_year,maker,state,rto_code,rto,vehicle_category,vehicle_model,fuel,vehicle_class,sale_type,vehicle_count,ingestion_timestamp,registration_month,registration_month_number,is_ev,ev_type
2023,HYUNDAI MOTOR INDIA LTD,Punjab,PB19,RTO BARNALA,LIGHT MOTOR VEHICLE,CRETA 1.5 CRDI MT EX,DIESEL,Motor Car,Resale,1,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2022,TVS MOTOR COMPANY LTD,Odisha,OR26,NUAPADA RTO,TWO WHEELER(NT),TVS RADEON,PETROL,M-Cycle/Scooter,New,3,2026-10-06T17:16:18.405Z,Mar,3,0,NON_EV
2020,TOYOTA KIRLOSKAR MOTOR PVT LTD,Karnataka,KA63,DHARWAD EAST RTO,LIGHT MOTOR VEHICLE,TOYOTA URBAN CRUISER (HIGH MT),PETROL,Motor Car,New,2,2026-10-06T17:16:18.405Z,Dec,12,0,NON_EV
2022,MARUTI SUZUKI INDIA LTD,Chhattisgarh,CG14,Jashpur DTO,LIGHT MOTOR VEHICLE,SWIFT LXI,PETROL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TATA MOTORS PASSENGER VEHICLES LTD,Karnataka,KA51,ELECTRONIC CITY RTO,LIGHT MOTOR VEHICLE,HARRIER XZ+ 2LBS6PH2 #DRK,DIESEL,Motor Car,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2021,HERO ELECTRIC VEHICLES PVT. LTD,Rajasthan,RJ14,JAIPUR (FIRST) RTO,TWO WHEELER(NT),OPTIMA HS 500 ER,ELECTRIC(BOV),M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Jan,1,1,BATTERY_EV
2021,BAJAJ AUTO LTD,Maharashtra,MH40,NAGPUR (RURAL),THREE WHEELER(T),BAJAJ MAXIMA C,DIESEL,Three Wheeler (Goods),Resale,1,2026-10-06T17:16:18.405Z,Jan,1,0,NON_EV
2020,SUZUKI MOTORCYCLE INDIA PVT LTD,Maharashtra,MH28,BULDHANA,TWO WHEELER(NT),ACCESS 125 DISC(ALLOY WHEEL),PETROL,M-Cycle/Scooter,Resale,1,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV
2023,HONDA MOTORCYCLE AND SCOOTER INDIA (P) LTD,Bihar,BR39,KATIHAR,TWO WHEELER(NT),ACTIVA SMART,PETROL,M-Cycle/Scooter,New,1,2026-10-06T17:16:18.405Z,Sep,9,0,NON_EV
2023,TVS MOTOR COMPANY LTD,Arunachal Pradesh,AR01,ITANAGAR CAPITAL REGION,TWO WHEELER(NT),TVS NTORQ 125 XT,PETROL,M-Cycle/Scooter,New,14,2026-10-06T17:16:18.405Z,Oct,10,0,NON_EV


## Aggregate by State

In [0]:
gold_state_df = (
    gold_state_source_df
    .groupBy("state")
    .agg(
        sum("vehicle_count").alias("total_registrations"),
        sum(
            when(
                col("is_ev") == 1, col("vehicle_count")
            ).otherwise(0)
        ).alias("ev_registrations")
    )
)
display(
    gold_state_df
    .orderBy(col("total_registrations").desc())
)


state,total_registrations,ev_registrations
Uttar Pradesh,404778,26092
Maharashtra,306897,17949
Tamil Nadu,217428,9751
Karnataka,189281,14251
Gujarat,188640,5396
Madhya Pradesh,178714,8093
Rajasthan,169896,8501
Bihar,157440,8341
West Bengal,127615,3133
Andhra Pradesh,111752,4131


## Calculate Non-EV Registration

In [0]:
gold_state_df = gold_state_df.withColumn(
    "non_ev_registrations",
    col("total_registrations") - col("ev_registrations")
)

display(
    gold_state_df
    .orderBy(col("total_registrations").desc())
)

state,total_registrations,ev_registrations,non_ev_registrations
Uttar Pradesh,404778,26092,378686
Maharashtra,306897,17949,288948
Tamil Nadu,217428,9751,207677
Karnataka,189281,14251,175030
Gujarat,188640,5396,183244
Madhya Pradesh,178714,8093,170621
Rajasthan,169896,8501,161395
Bihar,157440,8341,149099
West Bengal,127615,3133,124482
Andhra Pradesh,111752,4131,107621


##Calculate EV Penetration %

In [0]:
gold_state_df = gold_state_df.withColumn(
    "ev_penetration_pct",
    round(
        when(
            col("total_registrations") > 0,
            (
                col("ev_registrations") / 
                col("total_registrations")
            ) * 100
        ).otherwise(0),
        2
    )
)

display(
    gold_state_df
    .orderBy(col("ev_penetration_pct").desc())
)

state,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct
Goa,8535,863,7672,10.11
Tripura,6584,616,5968,9.36
Delhi,72772,6743,66029,9.27
Assam,68383,5509,62874,8.06
Chandigarh,6725,521,6204,7.75
Karnataka,189281,14251,175030,7.53
Kerala,94480,6497,87983,6.88
Uttar Pradesh,404778,26092,378686,6.45
Maharashtra,306897,17949,288948,5.85
Odisha,85395,4739,80656,5.55


## Calculate EV Market Share %

In [0]:
total_ev_window = Window.rowsBetween(
    Window.unboundedPreceding,
    Window.unboundedFollowing
)

gold_state_df = gold_state_df.withColumn(
    "total_ev_registrations",
    sum("ev_registrations").over(total_ev_window)
)

gold_state_df = gold_state_df.withColumn(
    "ev_market_share_pct",
    round(
        when(
            col("total_ev_registrations") > 0,
            (
                col("ev_registrations") / 
                col("total_ev_registrations")
            ) * 100
        ).otherwise(0),
        2
    )
)

gold_state_df = gold_state_df.drop("total_ev_registrations")

display(
    gold_state_df
    .orderBy(col("ev_market_share_pct").desc())
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


state,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct,ev_market_share_pct
Uttar Pradesh,404778,26092,378686,6.45,18.14
Maharashtra,306897,17949,288948,5.85,12.48
Karnataka,189281,14251,175030,7.53,9.91
Tamil Nadu,217428,9751,207677,4.48,6.78
Rajasthan,169896,8501,161395,5.0,5.91
Bihar,157440,8341,149099,5.3,5.8
Madhya Pradesh,178714,8093,170621,4.53,5.63
Delhi,72772,6743,66029,9.27,4.69
Kerala,94480,6497,87983,6.88,4.52
Assam,68383,5509,62874,8.06,3.83


## Final Column Order

In [0]:
gold_state_df = gold_state_df.select(
    "state",
    "total_registrations",
    "ev_registrations",
    "non_ev_registrations",
    "ev_penetration_pct",
    "ev_market_share_pct"
)

gold_state_df.printSchema()

display(
    gold_state_df
    .orderBy(col("ev_registrations").desc())
)

root
 |-- state: string (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registrations: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_market_share_pct: double (nullable = true)



state,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct,ev_market_share_pct
Uttar Pradesh,404778,26092,378686,6.45,18.14
Maharashtra,306897,17949,288948,5.85,12.48
Karnataka,189281,14251,175030,7.53,9.91
Tamil Nadu,217428,9751,207677,4.48,6.78
Rajasthan,169896,8501,161395,5.0,5.91
Bihar,157440,8341,149099,5.3,5.8
Madhya Pradesh,178714,8093,170621,4.53,5.63
Delhi,72772,6743,66029,9.27,4.69
Kerala,94480,6497,87983,6.88,4.52
Assam,68383,5509,62874,8.06,3.83


## Validate Totals

In [0]:
gold_state_df.selectExpr(
    "sum(total_registrations) as total_registrations",
    "sum(ev_registrations) as total_ev_registrations",
    "sum(non_ev_registrations) as total_non_ev_registrations"
).show()

+-------------------+----------------------+--------------------------+
|total_registrations|total_ev_registrations|total_non_ev_registrations|
+-------------------+----------------------+--------------------------+
|            2806654|                143873|                   2662781|
+-------------------+----------------------+--------------------------+



## Write Gold Table

In [0]:
gold_state_table = "ev_registration.gold.gold_ev_state"

gold_state_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable(gold_state_table)

## Verify

In [0]:
final_gold_state_df = spark.read.table(gold_state_table)

print("Gold state records:" , final_gold_state_df.count())

final_gold_state_df.printSchema()

display(
    final_gold_state_df
    .orderBy(col("ev_registrations").desc())
)

Gold state records: 35
root
 |-- state: string (nullable = true)
 |-- total_registrations: long (nullable = true)
 |-- ev_registrations: long (nullable = true)
 |-- non_ev_registrations: long (nullable = true)
 |-- ev_penetration_pct: double (nullable = true)
 |-- ev_market_share_pct: double (nullable = true)



state,total_registrations,ev_registrations,non_ev_registrations,ev_penetration_pct,ev_market_share_pct
Uttar Pradesh,404778,26092,378686,6.45,18.14
Maharashtra,306897,17949,288948,5.85,12.48
Karnataka,189281,14251,175030,7.53,9.91
Tamil Nadu,217428,9751,207677,4.48,6.78
Rajasthan,169896,8501,161395,5.0,5.91
Bihar,157440,8341,149099,5.3,5.8
Madhya Pradesh,178714,8093,170621,4.53,5.63
Delhi,72772,6743,66029,9.27,4.69
Kerala,94480,6497,87983,6.88,4.52
Assam,68383,5509,62874,8.06,3.83
